# CNN Baseline

This notebook uses the shared preprocessing notebook before defining the model.

In [ ]:
from pathlib import Path
from IPython import get_ipython

preprocessing_notebook = Path("shared/preprocessing.ipynb")
if not preprocessing_notebook.exists():
    preprocessing_notebook = Path("../shared/preprocessing.ipynb")

get_ipython().run_line_magic("run", str(preprocessing_notebook))

# Available for the model below:
# train_loader, val_loader, test_loader, pos_weight

# Model Architecture

In [ ]:
class ConvBlock(nn.Module):
    """Conv -> BatchNorm -> ReLU, repeated twice."""

    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

In [ ]:
class CNNBaseline(nn.Module):
    """Fully-convolutional encoder-decoder with NO skip connections.

    Channel progression: 12 -> 32 -> 64 -> 128 -> 256 -> 512 (bottleneck)
    -> 256 -> 128 -> 64 -> 32 -> 1

    This matches the team's U-Net in depth/width so that comparing
    CNNBaseline vs. UNet isolates the effect of skip connections alone.
    """

    def __init__(self, in_channels: int = 12, base_channels: int = 32, dropout: float = 0.2):
        super().__init__()
        c = base_channels  # 32

        # Encoder
        self.enc1 = ConvBlock(in_channels, c)          # 64x64 -> 64x64,  12->32
        self.enc2 = ConvBlock(c, c * 2)                # 32x32,           32->64
        self.enc3 = ConvBlock(c * 2, c * 4)             # 16x16,           64->128
        self.enc4 = ConvBlock(c * 4, c * 8)             # 8x8,             128->256
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = ConvBlock(c * 8, c * 16, dropout=dropout)  # 4x4, 256->512

        # Decoder (NO concatenation with encoder features — the key
        # architectural difference vs. U-Net)
        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(c * 8, c * 8, dropout=dropout)

        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(c * 4, c * 4, dropout=dropout)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(c * 2, c * 2)

        self.up1 = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(c, c)

        self.out_conv = nn.Conv2d(c, 1, kernel_size=1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        e1 = self.enc1(x)                 # (B, 32, 64, 64)
        e2 = self.enc2(self.pool(e1))      # (B, 64, 32, 32)
        e3 = self.enc3(self.pool(e2))      # (B, 128, 16, 16)
        e4 = self.enc4(self.pool(e3))      # (B, 256, 8, 8)

        b = self.bottleneck(self.pool(e4))  # (B, 512, 4, 4)

        d4 = self.dec4(self.up4(b))         # (B, 256, 8, 8)  -- no concat with e4
        d3 = self.dec3(self.up3(d4))        # (B, 128, 16, 16) -- no concat with e3
        d2 = self.dec2(self.up2(d3))        # (B, 64, 32, 32)  -- no concat with e2
        d1 = self.dec1(self.up1(d2))        # (B, 32, 64, 64)  -- no concat with e1

        return self.out_conv(d1)            # (B, 1, 64, 64) raw logits


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Quick sanity check on architecture shape
_test_model = CNNBaseline(in_channels=NUM_CHANNELS)
_test_out = _test_model(torch.randn(2, NUM_CHANNELS, 64, 64))
assert _test_out.shape == (2, 1, 64, 64), f"Unexpected output shape: {_test_out.shape}"
print(f"Architecture OK. Output shape: {tuple(_test_out.shape)}")
print(f"Total trainable parameters: {count_params(_test_model):,}")
del _test_model, _test_out